https://pmc.ncbi.nlm.nih.gov/articles/PMC12225070/

Paper reports
**MLP — accuracy = 0.937**,

## COPD (Wang et al., 2025)
**Reference**: Wang et al. "Construction and validation of a risk prediction model for COPD." 
BMC Pulmonary Medicine, 2025.

**Label**: Self-reported emphysema (MCQ160G), chronic bronchitis (MCQ160K), or COPD (MCQ160O)

**Inclusion criteria**: Adults ≥18 years

**Deviations from reference study**:

- Reference restricted to NHANES 2009-2018; we include all cycles except 2017-2020 (excluded due to missing MCQ variables)
- 5-fold stratified CV consistent with benchmark protoco

In [2]:
import pandas as pd
import numpy as np
import os, pickle

full = pd.read_pickle("/Users/alva/Documents/nhanes_tasks/data/NHANES/NHANES_master.pkl")

In [3]:
# --- COPD task ---
# Reference: Wang et al. (2025), BMC Pulmonary Medicine
# Population: Adults ≥18, NHANES 2009-2018
# Label: Self-reported emphysema, chronic bronchitis, or COPD diagnosis
# Deviation: All available cycles included; 5-fold stratified CV (same as reference)

subset_copd = full[
    (full['RIDAGEYR'] >= 18) &
    (full['YEAR'] != '2017-2020')
].copy()

subset_copd['COPD'] = (
    (subset_copd['MCQ160G'] == 1) |
    (subset_copd['MCQ160K'] == 1) |
    (subset_copd['MCQ160O'] == 1)
).astype(int)

def calc_smoking_years(row):
    if row['SMQ020'] != 1:
        return 0
    age_started = row['SMD030']
    if pd.isna(age_started):
        return np.nan
    if row['SMQ040'] in [1, 2]:
        return max(0, row['RIDAGEYR'] - age_started)
    age_stopped = row['SMD055']
    if pd.isna(age_stopped):
        return np.nan
    return max(0, age_stopped - age_started)

subset_copd['smoking_years'] = subset_copd.apply(calc_smoking_years, axis=1)
subset_copd['NLR'] = subset_copd['LBXNEPCT'] / subset_copd['LBXLYPCT']

features_copd = [
    'RIDAGEYR', 'BMXBMI', 'DMDMARTL', 'DMDEDUC2', 'smoking_years', 'NLR',
    'LBXSAL', 'LBXSCLSI', 'LBXSGL', 'LBXSIR', 'LBXSKSI', 'LBXSTB',
    'LBXSTP', 'LBXSTR', 'LBXSUA', 'LBXWBCSI', 'LBDMONO', 'LBXEOPCT',
    'LBXBAPCT', 'LBXRDW', 'LBDRFO', 'MCQ160A', 'MCQ160C', 'BPQ020', 'DIQ010'
]

data_copd = subset_copd[features_copd + ['COPD']].dropna(subset=['COPD'])
X_copd = data_copd[features_copd].values
y_copd = data_copd['COPD'].values

print(f"COPD: n={len(y_copd)}, prevalence={y_copd.mean():.3f}, features={X_copd.shape[1]}")

COPD: n=59204, prevalence=0.071, features=25


In [4]:
os.makedirs('/Users/alva/Documents/nhanes_tasks/data/tasks/', exist_ok=True)

task = {
    'X': X_copd,
    'y': y_copd,
    'features': features_copd,
    'name': 'copd'
}

with open('/Users/alva/Documents/nhanes_tasks/data/tasks/copd.pkl', 'wb') as f:
    pickle.dump(task, f)

print("Saved.")

Saved.
